<a href="https://colab.research.google.com/github/TienPhap0102/01-Banking-Transaction-Analysis/blob/main/Banking_Transaction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Banking Transaction Project**

## **1. Set up**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

## **2. Data Processing**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_bank = pd.read_excel('/content/drive/MyDrive/1. DATA ANH TÚ/Banking Transaction Project/Data/Banking_Transactional_Dataset.xlsx')
df_bank.head(2)

,TransactionID,CustomerID,TransactionDate,TransactionType,Amount,ProductCategory,ProductSubcategory,BranchCity,BranchLat,BranchLong,Channel,Currency,CreditCardFees,InsuranceFees,LatePaymentAmount,CustomerScore,MonthlyIncome,CustomerSegment,RecommendedOffer
0,1,8270,2025-01-29,Card Payment,6980.185223,Checking Account,Gold,Seville,37.3891,-5.9845,Branch,EUR,0.0,0.0,0.0,839,5767.68,Middle Income Segment,Mid-tier Savings Booster
1,2,1860,2023-02-10,Deposit,10786.371854,Mortgage,Gold,Murcia,37.9847,-1.1287,Branch,EUR,0.0,0.0,0.0,683,2441.00,Low Income Segment,Financial Literacy Program Access


In [ ]:
# Data overview
df_bank.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   TransactionID       20000 non-null  int64         
 1   CustomerID          20000 non-null  int64         
 2   TransactionDate     20000 non-null  datetime64[ns]
 3   TransactionType     20000 non-null  object        
 4   Amount              20000 non-null  float64       
 5   ProductCategory     20000 non-null  object        
 6   ProductSubcategory  20000 non-null  object        
 7   BranchCity          20000 non-null  object        
 8   BranchLat           20000 non-null  float64       
 9   BranchLong          20000 non-null  float64       
 10  Channel             20000 non-null  object        
 11  Currency            20000 non-null  object        
 12  CreditCardFees      20000 non-null  float64       
 13  InsuranceFees       20000 non-null  float64   

In [ ]:
# Check duplicate
df_bank.duplicated().sum()

np.int64(0)

In [ ]:
df_bank['Currency'].value_counts()

,count
Currency,
EUR,16974
USD,3026


In [ ]:
df_bank['BranchCity'].value_counts()

,count
BranchCity,
Murcia,2564
Barcelona,2564
Malaga,2524
Seville,2513
Zaragoza,2476
Madrid,2472
Bilbao,2455
Valencia,2432


**Key Findings:**

* The dataset contains **20.000 records**.

* There are **no duplicate and null** records within the table.

* Overall, the dataset is clean and aligns strictly with real-world business logic. The only trace of **"noise"** identified during the initial EDA phase is the inconsistency within the Currency column (**a mixture of USD and EUR**)

$\rightarrow$ **Action Plan:** It is mandatory to implement a Data Pre-processing step. **We must apply a fixed exchange rate to normalize all currency-related columns into a single Base Currency (EUR)**. This ensures absolute data integrity and accuracy before calculating any aggregated revenue or cost figures.

In [ ]:
# Exchange rate declaration
EXCHANGE_RATE_EUR_TO_USD = 1.16

# Create a list of all columns containing currency values ​​to be converted.
money_columns = ['Amount', 'CreditCardFees', 'InsuranceFees', 'LatePaymentAmount', 'MonthlyIncome']

# Use a loop to create new columns with the suffix '_EUR'.
for col in money_columns:
    new_col_name = f'{col}_EUR'
    df_bank[new_col_name] = np.where(df_bank['Currency'] == 'USD',
                                df_bank[col] / EXCHANGE_RATE_EUR_TO_USD,
                                df_bank[col]).round(2)

print("--- ALL CASH FLOW COLUMNS HAVE BEEN STANDARDIZED ---")
# Check
print(df_bank[['Currency', 'Amount_EUR', 'CreditCardFees_EUR', 'InsuranceFees_EUR','LatePaymentAmount_EUR','MonthlyIncome_EUR']].head())

--- ALL CASH FLOW COLUMNS HAVE BEEN STANDARDIZED ---
  Currency  Amount_EUR  CreditCardFees_EUR  InsuranceFees_EUR  \
0      EUR     6980.19                 0.0               0.00   
1      EUR    10786.37                 0.0               0.00   
2      EUR     3982.76                 0.0              92.46   
3      EUR    12408.64                 0.0               0.00   
4      USD     1610.57                 0.0               0.00   

   LatePaymentAmount_EUR  MonthlyIncome_EUR  
0                    0.0            5767.68  
1                    0.0            2441.00  
2                    0.0            9957.08  
3                    0.0            1545.80  
4                    0.0            5021.78  


In [ ]:
df_bank.head(2)

,TransactionID,CustomerID,TransactionDate,TransactionType,Amount,ProductCategory,ProductSubcategory,BranchCity,BranchLat,BranchLong,...,LatePaymentAmount,CustomerScore,MonthlyIncome,CustomerSegment,RecommendedOffer,Amount_EUR,CreditCardFees_EUR,InsuranceFees_EUR,LatePaymentAmount_EUR,MonthlyIncome_EUR
0,1,8270,2025-01-29,Card Payment,6980.185223,Checking Account,Gold,Seville,37.3891,-5.9845,...,0.0,839,5767.68,Middle Income Segment,Mid-tier Savings Booster,6980.19,0.0,0.0,0.0,5767.68
1,2,1860,2023-02-10,Deposit,10786.371854,Mortgage,Gold,Murcia,37.9847,-1.1287,...,0.0,683,2441.00,Low Income Segment,Financial Literacy Program Access,10786.37,0.0,0.0,0.0,2441.00


In [ ]:
df_bank['TransactionType'].value_counts()

,count
TransactionType,
Withdrawal,3395
Loan Payment,3369
Card Payment,3345
Fee,3308
Transfer,3293
Deposit,3290


In [ ]:
df_bank['ProductCategory'].value_counts()

,count
ProductCategory,
Credit Card,4082
Savings Account,4042
Loan,3998
Mortgage,3990
Checking Account,3888


In [ ]:
df_bank['ProductSubcategory'].value_counts()

,count
ProductSubcategory,
Student,4098
Business,4072
Platinum,3965
Gold,3956
Standard,3909


In [ ]:
df_bank['RecommendedOffer'].value_counts()

,count
RecommendedOffer,
Mid-tier Savings Booster,5220
Premium Investment Services,3954
Financial Literacy Program Access,3540
Exclusive Platinum Package,2672
Gold Card with Travel Benefits,1837
Personal Loan Cashback Offer,1828
No-Fee Basic Account,949


In [ ]:
min_date = df_bank['TransactionDate'].min()
max_date = df_bank['TransactionDate'].max()
print(f"Data Time Period: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")

Data Time Period: 2023-01-01 to 2025-05-20


## **3. Key Performance Metric**

In [ ]:
# 1. Operational Scale Metrics
total_transactions = df_bank['TransactionID'].nunique()
total_customers = df_bank['CustomerID'].nunique()

# 2. Transaction Volume Metrics
total_volume = df_bank['Amount_EUR'].sum()
atv = df_bank['Amount_EUR'].mean()

# 3. Fee Revenue Metrics
total_cc_fees = df_bank['CreditCardFees_EUR'].sum()
total_ins_fees = df_bank['InsuranceFees_EUR'].sum()
total_late_fees = df_bank['LatePaymentAmount_EUR'].sum()
total_revenue = total_cc_fees + total_ins_fees + total_late_fees

# PRINTING THE BUSINESS OVERVIEW DASHBOARD
print("📊 BUSINESS OVERVIEW REPORT")
print("-" * 50)
print(f"👥 Total Unique Customers: {total_customers:,.0f}")
print(f"🔄 Total Transactions: {total_transactions:,.0f}")
print("-" * 50)
print(f"💰 Total Transaction Volume: {total_volume:,.2f} EUR")
print(f"💳 Average Transaction Value (ATV): {atv:,.2f} EUR")
print("-" * 50)
print(f"📈 TOTAL FEE REVENUE: {total_revenue:,.2f} EUR")
print(f"   - Credit Card Fees: {total_cc_fees:,.2f} EUR")
print(f"   - Insurance Fees: {total_ins_fees:,.2f} EUR")
print(f"   - Late Payment Fees: {total_late_fees:,.2f} EUR")
print("-" * 50)

📊 BUSINESS OVERVIEW REPORT
--------------------------------------------------
👥 Total Unique Customers: 8,025
🔄 Total Transactions: 20,000
--------------------------------------------------
💰 Total Transaction Volume: 98,898,129.50 EUR
💳 Average Transaction Value (ATV): 4,944.91 EUR
--------------------------------------------------
📈 TOTAL FEE REVENUE: 623,930.41 EUR
   - Credit Card Fees: 102,229.24 EUR
   - Insurance Fees: 195,356.22 EUR
   - Late Payment Fees: 326,344.95 EUR
--------------------------------------------------


**Key findings:**

Based on the **29-month** transaction data, the bank's operational footprint **reflects a high-value**, **low-frequency** business model:

* **Transaction Volume & Client Profiling:**

  * The system processed a massive Total Volume of **98.89 Million EUR** across a relatively concentrated base of **8,025 unique customers**.

  * The exceptionally **high Average Transaction Value** (~ 4,945 EUR/transaction) combined with **low engagement** (~ 2.5 transactions/user) strongly indicates that this portfolio consists of High-Net-Worth Individuals, SME corporate accounts, or episodic large-ticket financial events (mortgage settlements, loan disbursements, or annual insurance premiums) rather than day-to-day retail banking.

* **Fee Revenue Structure (Non-Interest Income):**

  * Total Fee Revenue stands at **~624K EUR**, representing just **0.63% of the total transaction volume**. This aligns with standard banking models where the primary revenue stream is Interest Margin, making these fees a supplementary income source.

  * **Credit Risk Indicator:** Late Payment Fees dominate the fee structure, **driving 52.3% of this revenue**. Rather than a purely negative customer experience metric, this indicates a specific credit risk profile within the portfolio. **A significant portion of the user base relies on short-term liquidity extensions and incurs penalty charges**.

## **4. EDA**

In [ ]:

# DESCRIPTIVE STATISTICS

# Select the columns of standardized numbers for EUR
cols_to_describe = ['Amount_EUR', 'MonthlyIncome_EUR', 'CreditCardFees_EUR',
                    'InsuranceFees_EUR', 'LatePaymentAmount_EUR', 'CustomerScore']

# Summary Statistics
stats_summary = df_bank[cols_to_describe].describe().transpose()

pd.set_option('display.float_format', '{:,.2f}'.format)
print("📊 SUMMARY STATISTICS")
print(stats_summary)

# Customer Segment
segment_stats = df_bank.groupby('CustomerSegment')[['Amount_EUR', 'CustomerScore']].agg(['mean', 'median', 'count'])
print("\n📊 SEGMENTATION INSIGHTS")
print(segment_stats)

📊 SUMMARY STATISTICS
                          count     mean      std    min      25%      50%  \
Amount_EUR            20,000.00 4,944.91 3,466.17   8.28 2,154.88 4,285.24   
MonthlyIncome_EUR     20,000.00 5,370.80 2,566.64 863.23 3,141.81 5,348.97   
CreditCardFees_EUR    20,000.00     5.11    11.99   0.00     0.00     0.00   
InsuranceFees_EUR     20,000.00     9.77    23.24   0.00     0.00     0.00   
LatePaymentAmount_EUR 20,000.00    16.32    43.26   0.00     0.00     0.00   
CustomerScore         20,000.00   575.30   159.42 300.00   437.00   577.00   

                           75%       max  
Amount_EUR            7,221.50 14,895.17  
MonthlyIncome_EUR     7,566.19  9,998.88  
CreditCardFees_EUR        0.00     49.97  
InsuranceFees_EUR         0.00     99.95  
LatePaymentAmount_EUR     0.00    199.98  
CustomerScore           715.00    849.00  

📊 SEGMENTATION INSIGHTS
                      Amount_EUR                CustomerScore             
                            mea

**Key findings:**
* The average transaction value is **4,944 EUR**; however, **the standard deviation is high** (3,466 EUR). This indicates **significant volatility**, suggesting that a subset of "**High-Ticket**" transactions is heavily skewing the average.

* For all fee categories, **the Median is 0**, implying that **over 50%** of transactions generate zero fee revenue. The bank's non-interest income is driven primarily by a specific "**long-tail**" segment of customers.

## **Export Data to Analysis**

In [ ]:
df_bank.to_csv('banking_transaction_data.csv', index=False)
print("Data exported to 'banking_transaction_data.csv' successfully!")

Data exported to 'banking_transaction_data.csv' successfully!
